# Day 2 - Toolbox & Evaluation - SOLUTIONS

> Instructor copy. Every TODO is filled in and every question answered.
> The student copy is the same notebook with these cells blanked.

## Setup

Same `coursekit` imports, plus `statsforecast` for the models and
`utilsforecast` for the metrics.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import (MSTL, HistoricAverage, Naive,
                                  RandomWalkWithDrift, SeasonalNaive)
from statsforecast.utils import ConformalIntervals
from statsmodels.stats.diagnostic import acorr_ljungbox
from utilsforecast.losses import mae, mape, mase, rmse, rmsse, scaled_crps

from coursekit import checks
from coursekit import datasets as D
from coursekit import leaderboard as lb
from coursekit import plotting as P

P.use_course_style()

spine = D.spine()
H = 24
train, test = D.train_test(spine, h=H)

# Every forecast today is asked for the same ladder of intervals. 80 is the one
# we read coverage off; the rest are there so Exercise 2.5 can score the whole
# forecast DISTRIBUTION and not just one band.
LEVELS = [20, 40, 60, 80, 95]

print(f"train: {len(train)} months to {train['ds'].max().date()}")
print(f"test : {len(test)} months from {test['ds'].min().date()}")

---
# Exercise 2.1 - The benchmark floor

*Follows segment 1. 13 minutes.*

Fit all four benchmarks and look at them. Everything for the rest of the course
is measured against these.

In [ ]:
MODELS = ["HistoricAverage", "Naive", "SeasonalNaive", "RWD"]
LABELS = {"HistoricAverage": "Mean", "Naive": "Naive",
          "SeasonalNaive": "Seasonal naive", "RWD": "Drift"}

BENCHMARKS = [HistoricAverage(), Naive(), SeasonalNaive(season_length=12),
              RandomWalkWithDrift()]

sf = StatsForecast(models=BENCHMARKS, freq=D.FREQ, n_jobs=1)
fc = sf.forecast(df=train, h=H, level=LEVELS, fitted=True)

checks.check_ex_2_1(fc, MODELS)
fc.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
hist = train.tail(72)
ax.plot(hist["ds"], hist["y"], color=P.BLACK, lw=1.1, label="observed")
ax.plot(test["ds"], test["y"], color=P.GREY, lw=1.4, ls="--", label="actual")
for m, c in zip(MODELS, [P.PINK, P.GREEN, P.ORANGE, P.BLUE]):
    ax.plot(fc["ds"], fc[m], lw=1.7, color=c, label=LABELS[m])
ax.set(title="Four benchmarks, 24 months ahead")
ax.legend(frameon=False, ncols=3)
plt.show()

**Question.** Two of these are obviously wrong before you compute a single
metric. Which, and why?


*Your answer:*


*Answer.* The **mean** method forecasts a flat line at roughly 160 for a series
currently sitting near 370 - it averages over 37 years of growth, so it is
hopeless on any trending series. The **naive** method forecasts a flat line at
the last value, which throws away the seasonality we spent all of Day 1
establishing. **Drift** at least captures the trend but still ignores season.
Only the **seasonal naive** reproduces the annual shape.

### A fifth model, out of Day 1

You already know how to take this series apart: STL gives you trend, season and
remainder. Ch 5.7 turns that into a *forecasting* method. Strip the season off,
forecast the seasonally adjusted series with something that handles trend -
drift, say - then add last year's seasonal shape back on top.

`MSTL` is that recipe in one object, and nothing in it is new to you.
`RandomWalkWithDrift` is a benchmark you fit ten minutes ago; the seasonal part
is a seasonal naive on the seasonal component.

In [ ]:
sf = StatsForecast(
    models=BENCHMARKS + [MSTL(season_length=12,
                              trend_forecaster=RandomWalkWithDrift())],
    freq=D.FREQ, n_jobs=1,
)
fc = sf.forecast(df=train, h=H, level=LEVELS, fitted=True)

MODELS = ["HistoricAverage", "Naive", "SeasonalNaive", "RWD", "MSTL"]
LABELS["MSTL"] = "STL + drift"
print(f"{len(MODELS)} models: {', '.join(LABELS[m] for m in MODELS)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
hist = train.tail(60)
ax.plot(hist["ds"], hist["y"], color=P.BLACK, lw=1.1, label="observed")
ax.plot(test["ds"], test["y"], color=P.GREY, lw=1.6, ls="--", label="actual")
ax.plot(fc["ds"], fc["SeasonalNaive"], color=P.ORANGE, lw=1.7, label="seasonal naive")
ax.plot(fc["ds"], fc["MSTL"], color=P.BLUE, lw=1.7, label="STL + drift")
ax.set(title="The decomposition route vs. the floor")
ax.legend(frameon=False, ncols=4)
plt.show()

**Question.** Which of those two tracks the holdout better, and what is the STL
route doing that the seasonal naive cannot? Write your answer down now - you
will be asked to revisit it in Exercise 2.5.


*Your answer:*


*Answer.* The STL route is clearly closer over these 24 months. The seasonal
naive repeats last year's level exactly, so on a series that grows about 6% a
year it starts the horizon low and stays low - the gap is a *bias*, visible as
the forecast sitting under the actuals almost everywhere. The STL route
separates that trend out and lets drift carry it forward, so it keeps the
seasonal shape *and* the growth.

Hold that conclusion loosely. It is one window.

### Stretch - forecasting on a transformed scale

The spine is multiplicative. Forecast the Box-Cox transformed series, then
back-transform. Note that the naive back-transform gives you the **median**, not
the mean.

In [ ]:
from coreforecast.scalers import boxcox, boxcox_lambda, inv_boxcox

lam = boxcox_lambda(train["y"].to_numpy(), method="loglik")
train_t = train.assign(y=boxcox(train["y"].to_numpy(), lam))

sf_t = StatsForecast(models=[SeasonalNaive(season_length=12)], freq=D.FREQ, n_jobs=1)
fc_t = sf_t.forecast(df=train_t, h=H, level=[80])
back = inv_boxcox(fc_t["SeasonalNaive"].to_numpy(), lam)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train.tail(48)["ds"], train.tail(48)["y"], color=P.BLACK, lw=1.1, label="observed")
ax.plot(test["ds"], test["y"], color=P.GREY, ls="--", lw=1.4, label="actual")
ax.plot(fc["ds"], fc["SeasonalNaive"], color=P.ORANGE, lw=1.6, label="SNaive, raw scale")
ax.plot(fc_t["ds"], back, color=P.BLUE, lw=1.6, label="SNaive, via Box-Cox")
ax.legend(frameon=False, ncols=2)
ax.set(title=f"Forecasting on the transformed scale (lambda = {lam:.3f})")
plt.show()

print("Back-transforming a forecast gives the MEDIAN of the forecast "
      "distribution, not the mean. For a skewed distribution those differ, and "
      "if you are adding forecasts up (across stores, across months) you need "
      "means. That correction is the 'bias adjustment' in Ch 5.6.")

---
# Exercise 2.2 - Are the residuals white noise?

*Follows segment 2. 13 minutes.*

If a model's residuals still carry structure, the model has not finished.

In [ ]:
fv = sf.forecast_fitted_values()

resid = fv["y"] - fv["SeasonalNaive"]
fig, axes = P.residual_diagnostics(resid, ds=fv["ds"],
                                   title="Seasonal naive residuals")
plt.show()

lb_pvalue = float(acorr_ljungbox(resid.dropna(), lags=[24],
                                 return_df=True)["lb_pvalue"].iloc[0])

print(f"mean residual : {pd.Series(resid).mean():.3f}")
print(f"Ljung-Box p   : {lb_pvalue:.3e}")

checks.check_ex_2_2(resid, lb_pvalue)

In [ ]:
resid_d = fv["y"] - fv["RWD"]
fig, axes = P.residual_diagnostics(resid_d, ds=fv["ds"], title="Drift residuals")
plt.show()

for name, r in [("SeasonalNaive", resid), ("RWD", resid_d)]:
    r = r.dropna()
    p = float(acorr_ljungbox(r, lags=[24], return_df=True)["lb_pvalue"].iloc[0])
    first_half, second_half = r.iloc[:len(r) // 2], r.iloc[len(r) // 2:]
    print(f"{name:<14} mean={r.mean():8.3f}  LB p={p:.2e}  "
          f"sd early={first_half.std():6.2f}  sd late={second_half.std():6.2f}")

**Write your verdict.** For each method, which of the four residual properties
hold, and what does that imply?


*Your answer:*


*Answer.* Neither is close to white noise.

- **Uncorrelated:** fails badly for both - Ljung-Box p is effectively zero and
  the residual ACF has large spikes. There is a great deal of signal left.
- **Zero mean:** the seasonal naive's mean residual is clearly positive, because
  the series trends upward and last year's value is systematically too low. That
  is a *bias*: the forecast will be low every time.
- **Constant variance:** fails - the late-period standard deviation is several
  times the early one, because the series grew eightfold. This is exactly what
  the Box-Cox transform in 1.4 addresses.
- **Normal:** roughly, but with heavy tails.

Implication: the benchmark floor is a floor, not a model. The failures are
informative - the bias says "add a trend", the seasonal spikes say "the seasonal
shape has changed", the variance says "transform first".

---
# Exercise 2.3 - Intervals, and how much to believe them

*Follows segment 3. 20 minutes.*

Three ways to draw an interval around the same point forecast, each spending a
different assumption: **Gaussian** (part a), **bootstrap** (part b) and
**conformal** (part c).

## Part a - the Gaussian interval

In [ ]:
fan = fc.rename(columns={
    "SeasonalNaive": "mean",
    "SeasonalNaive-lo-80": "lo-80", "SeasonalNaive-hi-80": "hi-80",
    "SeasonalNaive-lo-95": "lo-95", "SeasonalNaive-hi-95": "hi-95",
})
fig, ax = plt.subplots(figsize=(10, 4.6))
P.fan_chart(train, fan, levels=(80, 95), ax=ax, actual=test, history_tail=72,
            title="Seasonal naive with prediction intervals")
plt.show()

In [ ]:
width = fc["SeasonalNaive-hi-80"] - fc["SeasonalNaive-lo-80"]

h = np.arange(1, len(width) + 1)
k = (h - 1) // 12
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(h, width, color=P.BLUE, lw=4.5, alpha=0.55, label="actual width")
ax.plot(h, width.iloc[0] * np.sqrt(k + 1), color=P.GREEN, lw=2, dashes=(6, 4),
        label="width_1 * sqrt(k+1)")
ax.plot(h, width.iloc[0] * np.sqrt(h), color=P.ORANGE, ls=":", lw=1.4,
        label="width_1 * sqrt(h)")
ax.set(xlabel="horizon h", ylabel="80% interval width", title="Widening with h")
ax.legend(frameon=False)
plt.show()

print("The width is FLAT for h = 1..12, then steps up by sqrt(2). A seasonal "
      "naive reuses one year's residual spread, so sqrt(h) - which belongs to "
      "the naive method - overstates the width by 3.5x at h = 24.")

In [ ]:
merged = test.merge(fc, on=["unique_id", "ds"])

coverage = float(((merged["y"] >= merged["SeasonalNaive-lo-80"])
                  & (merged["y"] <= merged["SeasonalNaive-hi-80"])).mean())
se = float(np.sqrt(0.8 * 0.2 / len(merged)))

print(f"nominal 80%,  measured {coverage:.1%}  +/- {1.96 * se:.1%} (95% CI)")
checks.check_ex_2_3(width, coverage, se)

**Question.** Your measured coverage came with an error bar roughly 16 points
wide. What would you have to change to measure coverage properly - and is that
what exercise 2.5 does?


*Your answer:*


*Answer.* You need more scored points, and they must come from *different
origins* rather than from extending one test window (extending it just forecasts
further ahead, where the model is worse). Rolling-origin cross-validation gives
exactly that: 8 folds x 12 months = 96 scored points instead of 24, cutting the
standard error in half. That is exercise 2.5.

Note it does not fix the *other* problem - the interval formula ignores model
uncertainty - so even a well-measured coverage tends to come in under nominal.

---
## Part b - the same interval, without the normality assumption

The Gaussian interval spends three assumptions: uncorrelated residuals, constant
variance, and **normality**. The residual bootstrap buys the third one back. It
resamples the errors you actually saw:

$$y^*_{T+i} = y^*_{T+i-m} + e^*_{T+i}$$

where $e^*$ is drawn at random from the pool of past residuals. Run that
recursion a few thousand times and you have a few thousand possible futures; the
interval is a percentile taken down each column.

> **The assumption you just made.** The simple residual bootstrap assumes the
> residuals come from one common distribution $\hat{F}$ whose distributional
> characteristics **do not change over time** - i.i.d. draws from the pool of
> past errors. Hold on to that; part d comes back to it.

In [ ]:
resid_sn = (fv["y"] - fv["SeasonalNaive"]).dropna()
boot_paths = P.bootstrap_paths(train["y"], resid_sn, h=H, season_length=12,
                               n_paths=5000, seed=7)

fig, ax = plt.subplots(figsize=(10, 4))
P.sim_paths_plot(train, fc["ds"], boot_paths, ax=ax, n_show=8, history_tail=60,
                 actual=test, title="Eight of the 5000 simulated futures")
plt.show()

In [ ]:
boot_fan = P.paths_to_fan(fc["ds"], boot_paths, levels=(80, 95))

for lvl in (80, 95):
    g = float((fc[f"SeasonalNaive-hi-{lvl}"] - fc[f"SeasonalNaive-lo-{lvl}"]).mean())
    b = float((boot_fan[f"hi-{lvl}"] - boot_fan[f"lo-{lvl}"]).mean())
    print(f"mean {lvl}% width   gaussian {g:5.1f}   bootstrap {b:5.1f}")

print("\nThe bootstrap is about a fifth narrower at 80% but level with the "
      "Gaussian at 95%. Sharp peak, fat tails - which is exactly what an excess "
      "kurtosis of 5.7 looks like. Note also that it is not symmetric about the "
      "mean; nothing forced it to be.")

checks.check_ex_2_3b(boot_paths, boot_fan, fc)

---
## Part c - $e_{t+h|t}$, and conformal prediction

Conformal prediction throws away the distribution entirely and calibrates on
**$h$-step-ahead forecast errors**:

$$e_{t+h|t} = y_{t+h} - \hat{y}_{t+h|t}$$

- $t$ is when the forecast was **made** (the forecast origin)
- $h$ is the **horizon**, $t+h$ the time being predicted
- $y_{t+h}$ is what happened; $\hat{y}_{t+h|t}$ is the forecast made at $t$ for $t+h$

*Concrete:* you hold $y_1 \dots y_{10}$ and want a 3-step-ahead interval. Stand
at $t = 5$, forecast $\hat{y}_{8|5}$, then look up $y_8$ and record
$e_{8|5} = y_8 - \hat{y}_{8|5}$. Slide the origin to $t = 6, 7, \dots$ and
repeat. At $h = 1$ these are exactly the residuals from exercise 2.2; for
$h > 1$ they are a wider set that has to be **collected**, not fitted.

Build that collection yourself before letting `statsforecast` do it.

In [ ]:
cv12 = sf.cross_validation(df=train, h=12, step_size=12, n_windows=8)
cv12 = cv12[cv12["ds"] == cv12["cutoff"] + pd.DateOffset(months=12)]
e12 = cv12["y"] - cv12["SeasonalNaive"]

print("h = 12 errors:", np.round(np.asarray(e12), 1))
print(f"Q_0.80(|e|)  = {np.quantile(np.abs(e12), 0.80):.1f}"
      "   <- half-width of an 80% conformal interval at h = 12")

print("\nThat first error of +120.8 is not a bug: turnover went from 219.5 in "
      "Dec 2008 to 340.3 in Dec 2009 and never came back. Eight errors is a thin "
      "calibration set, and one break that size in it is exactly why a conformal "
      "band built from few windows comes out jittery.")

fig, ax = plt.subplots(figsize=(10, 4))
P.h_step_error_diagram(
    train.tail(96),
    cv12.rename(columns={"SeasonalNaive": "yhat"})[["cutoff", "ds", "y", "yhat"]],
    ax=ax, title="Eight origins, h = 12: the calibration set")
plt.show()

In [ ]:
sf_conf = StatsForecast(
    models=[SeasonalNaive(season_length=12,
                          prediction_intervals=ConformalIntervals(n_windows=8, h=H))],
    freq=D.FREQ, n_jobs=1,
)
fc_conf = sf_conf.forecast(df=train, h=H, level=[80, 95])

y_true = test["y"].to_numpy()
rows = []
for name, lo, hi in [
    ("Gaussian", fc["SeasonalNaive-lo-80"], fc["SeasonalNaive-hi-80"]),
    ("Bootstrap", boot_fan["lo-80"], boot_fan["hi-80"]),
    ("Conformal", fc_conf["SeasonalNaive-lo-80"], fc_conf["SeasonalNaive-hi-80"]),
]:
    lo, hi = np.asarray(lo), np.asarray(hi)
    rows.append({"method": name,
                 "width_80": float((hi - lo).mean()),
                 "coverage_80": float(((y_true >= lo) & (y_true <= hi)).mean())})
cmp = pd.DataFrame(rows)

checks.check_ex_2_3c(cmp)
cmp.round(3)

**Question.** Each method spends a different assumption. Name the assumption
each one makes, and say which of them **this series** breaks.


*Your answer:*


*Answer.*

| Method | Assumes | True here? |
|---|---|---|
| Gaussian | residuals uncorrelated, constant variance, **normal** | no, no, no |
| Bootstrap | residuals uncorrelated, **i.i.d. from $\hat{F}$** | no - the residual SD wanders between 4 and 35 |
| Conformal | past $h$-step errors **exchangeable** with future ones | closest, but a series whose error spread keeps growing is drifting, not exchangeable |

Exchangeability is the weakest of the three: it only asks that the order of the
past errors carries no information, not that they are independent or that they
follow any named distribution. That is why conformal survives this series best.

None of the three is *satisfied* here. The point is not to find a method with no
assumptions - there isn't one - but to know which assumption you are spending
and whether the data supports it.

### Stretch - the assumption is a knob

`P.bootstrap_paths(..., resid_tail=N)` draws only from the last `N` residuals.
If the residual distribution really were constant over time, that would just
throw information away. Sweep `N` and see.

In [ ]:
# 24 holdout points cannot resolve a coverage rate (exercise 2.3, part a), so
# score every pool size over 8 rolling origins instead - 96 points each.
sf_sn = StatsForecast(models=[SeasonalNaive(season_length=12)], freq=D.FREQ, n_jobs=1)

for tail in (60, 120, 180, None):
    inside, widths = [], []
    for w in range(8):
        end = len(spine) - (8 - w) * 12
        tr, te = spine.iloc[:end], spine.iloc[end:end + 12]
        sf_sn.forecast(df=tr, h=12, fitted=True)
        r = (sf_sn.forecast_fitted_values()
             .pipe(lambda d: d["y"] - d["SeasonalNaive"]).dropna())
        paths = P.bootstrap_paths(tr["y"], r, h=12, season_length=12,
                                  n_paths=5000, seed=7, resid_tail=tail)
        f = P.paths_to_fan(te["ds"], paths, levels=(80,))
        lo, hi = f["lo-80"].to_numpy(), f["hi-80"].to_numpy()
        inside.append((te["y"].to_numpy() >= lo) & (te["y"].to_numpy() <= hi))
        widths.append((hi - lo).mean())
    label = "all" if tail is None else f"last {tail}"
    print(f"pool = {label:>8} residuals   mean width {np.mean(widths):5.1f}   "
          f"coverage {np.concatenate(inside).mean():5.1%}  (96 points)")

print("\nThe bootstrap covers only about 62% with the full 405-residual pool: "
      "pooling the early, low-spread residuals with the recent, high-spread ones "
      "makes the draw pool far too tight for a recent forecast. Coverage climbs "
      "monotonically as the pool gets shorter and more recent: about 75% with the "
      "last 180, about 87% with the last 120, about 91% with the last 60, "
      "overshooting the nominal 80% at the other end. If the residuals really did "
      "come from one unchanging distribution, throwing away the older ones could "
      "only make the estimate noisier, never systematically better. That trend IS "
      "the identically-distributed assumption failing, and the pool length is the "
      "knob you have for it.")

---
# Exercise 2.4 - Scoring, and the metric that lies

*Follows segment 4. 12 minutes.*

In [ ]:
scores = pd.DataFrame({
    "MAE": mae(merged, models=MODELS)[MODELS].iloc[0],
    "RMSE": rmse(merged, models=MODELS)[MODELS].iloc[0],
    "MAPE_pct": mape(merged, models=MODELS)[MODELS].iloc[0] * 100,
    "MASE": mase(merged, models=MODELS, seasonality=12, train_df=train)[MODELS].iloc[0],
    "RMSSE": rmsse(merged, models=MODELS, seasonality=12, train_df=train)[MODELS].iloc[0],
})
scores.index = [LABELS[m] for m in scores.index]

checks.check_ex_2_4(scores)
scores.round(3)

Now build the case where MAPE misleads. Construct a near-zero series and two
forecasts: one that is a little too **low**, one that is much too **high**.

In [ ]:
rng = np.random.default_rng(3)
n = 48
low = pd.DataFrame({
    "ds": pd.date_range("2020-01-01", periods=n, freq="MS"),
    "y": np.clip(rng.poisson(1.4, n).astype(float), 0.2, None),
})

pred_hi = low["y"] + 2.0
pred_lo = low["y"] - 0.15
for name, pred in [("A: +2.00 units", pred_hi), ("B: -0.15 units", pred_lo)]:
    err = low["y"] - pred
    print(f"{name:<18} MAE = {err.abs().mean():5.2f}   "
          f"MAPE = {(err / low['y']).abs().mean() * 100:8.1f}%")

print("\nMAE says B is 13x better, which matches the picture. MAPE agrees on "
      "direction here but wildly exaggerates: dividing a 2-unit error by an "
      "actual of 0.2 gives 1000%. On a series that ever touches zero, MAPE is "
      "undefined outright.")

**Rank the four benchmarks and defend the ranking.** Which metric did you use,
and why not the others?


*Your answer:*


*Answer.* STL + drift > Seasonal naive > Naive > Drift > Mean, on MASE.

MASE, because it is scale-free (so this ranking can be compared against other
series later), it is defined even when the series touches zero, and the
benchmark is built into it - the seasonal naive's MASE of 1.11 immediately tells
you it is still slightly worse than a one-step seasonal naive, while the STL
route's 0.70 says it clears that bar comfortably.

Not MAE or RMSE: correct here, but their units are millions of dollars, so they
cannot be pooled across series. Not MAPE: this series never approaches zero so
it happens to behave, but selecting on MAPE builds a habit that breaks the first
time you meet slow-moving demand.

Note what you have just done: picked a winner off **one** 24-month window. That
is the exact move Exercise 2.5 is about to take apart.

---
# Exercise 2.5 - The harness

*Follows segment 5. 17 minutes.*

This is the exercise the rest of the course rests on. You are building the
evaluation harness that every Day 3 model gets plugged into.

In [ ]:
cv = sf.cross_validation(df=spine, h=12, step_size=12, n_windows=8, level=LEVELS)

print(f"folds : {cv['cutoff'].nunique()}")
print(f"scored points : {len(cv)}")
cv.head()

Coverage answers one question - *is the 80% band honest?* - and it is blind to
everything else. An interval of plus-or-minus infinity has perfect coverage and
is worth nothing, and two models that both cover 80% can have wildly different
widths. To *rank* forecast distributions you need a proper score.

`scaled_crps` is that score: it averages the quantile (pinball) loss over the
whole ladder of `LEVELS`, so being too wide, too narrow, or centred in the wrong
place all cost you, in one scale-free number. Lower is better.

In [ ]:
# Which quantile each of those interval columns actually is, low to high.
QUANTILES = np.array([0.025, 0.10, 0.20, 0.30, 0.40, 0.60, 0.70, 0.80, 0.90, 0.975])
QCOLS = ["lo-95", "lo-80", "lo-60", "lo-40", "lo-20",
         "hi-20", "hi-40", "hi-60", "hi-80", "hi-95"]


def qcols(model):
    """The ten interval columns of one model, in QUANTILES order."""
    return [f"{model}-{c}" for c in QCOLS]


print(qcols("SeasonalNaive"))

In [ ]:
rows = []
for m in MODELS:
    fold_mase, fold_rmsse = [], []
    for cut, g in cv.groupby("cutoff"):
        tr = spine[spine["ds"] <= cut]
        g1 = g.drop(columns=["cutoff"])
        fold_mase.append(mase(g1, models=[m], seasonality=12, train_df=tr)[m].iloc[0])
        fold_rmsse.append(rmsse(g1, models=[m], seasonality=12, train_df=tr)[m].iloc[0])
    crps = scaled_crps(cv.drop(columns=["cutoff"]), models={m: qcols(m)},
                       quantiles=QUANTILES)[m].iloc[0]
    inside = ((cv["y"] >= cv[f"{m}-lo-80"]) & (cv["y"] <= cv[f"{m}-hi-80"])).mean()
    rows.append({"model": LABELS[m], "mase": np.mean(fold_mase),
                 "rmsse": np.mean(fold_rmsse), "crps": float(crps),
                 "coverage_80": float(inside),
                 "mase_min": np.min(fold_mase), "mase_max": np.max(fold_mase)})

summary = pd.DataFrame(rows)

checks.check_ex_2_5(cv, summary)
summary.round(3)

**Go back and read your answer from Exercise 2.1.** On the single 24-month
window the STL route beat the seasonal naive on MASE, 0.70 to 1.11. What does
the table above say, and which of the two numbers would you put in front of a
stakeholder?


*Your answer:*


*Answer.* Across eight origins the ordering **flips**: the seasonal naive
averages about 1.18 and the STL route about 1.22, and the seasonal naive wins on
scaled CRPS too. The single window was not a lie - the STL route really was
better over those particular 24 months - it was just one draw from a
distribution wide enough to contain both answers.

The number to report is the eight-fold one, with its spread. The single-window
0.70 is exactly the kind of result that gets a model promoted into production on
the strength of a lucky year.

Note also that the STL route earns its worse CRPS with *narrower* intervals
(about 40 units wide against the seasonal naive's 49) and worse coverage
(about 61% against 77%). Narrow is not the same as good: CRPS charges you for
the misses that narrowness buys, which is precisely what coverage on its own
cannot tell you.

Write the results to the leaderboard. **This file is the course's running
scoreboard** - Day 3 appends to the same table.

In [ ]:
lb.reset()   # start clean; re-running this cell is safe

for _, row in summary.iterrows():
    lb.record(
        row["model"], day=2,
        mase=float(row["mase"]), rmsse=float(row["rmsse"]),
        crps=float(row["crps"]), coverage_80=float(row["coverage_80"]),
        notes="Day 2 baseline, 8-fold rolling origin",
    )

table = lb.show()
checks.check_leaderboard(table)
table.round(3)

### Stretch - how much does one window matter?

Score each fold separately and look at the spread.

In [ ]:
per_fold = pd.DataFrame([
    mase(g.drop(columns=["cutoff"]), models=MODELS, seasonality=12,
         train_df=spine[spine["ds"] <= cut])[MODELS].iloc[0]
    for cut, g in cv.groupby("cutoff")
]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 3.8))
for i, m in enumerate(MODELS):
    ax.scatter(np.full(len(per_fold), i), per_fold[m], s=45, color=P.ORANGE,
               alpha=0.75, zorder=3)
ax.set_xticks(range(len(MODELS)), [LABELS[m] for m in MODELS], rotation=20, ha="right")
ax.set(ylabel="MASE", title="One dot per fold")
ax.set_yscale("log")
plt.show()

sn = per_fold["SeasonalNaive"]
print(f"Seasonal naive MASE by fold: {'  '.join(f'{v:.2f}' for v in sn)}")
print(f"best {sn.min():.2f}, worst {sn.max():.2f} - a {sn.max() / sn.min():.1f}x spread")
print("\nThe RANKING was identical in every fold. The NUMBER was not. Report "
      "the ranking with confidence and the number with a spread.")

---
## End of Day 2

You have an evaluation harness: benchmarks, residual diagnostics, intervals with
an honest error bar, scale-free metrics, and rolling-origin cross-validation.

`labs/leaderboard.csv` now holds the benchmark floor. Every model on Day 3 has
to get past it.